In [1]:
# Import necessary libraries and modules
import pandas as pd
import pandapower as pp
import pandapower.networks as pn

from simulation_interface import GridSimulation
from typing import List
from pandapower import pandapowerNet
from sim_config import GridConfig, DataCenterConfig
from grid_regions import GridRegion
from grid_observer import plot_data
from datetime import datetime, timedelta
from generation_types import GenerationType

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

In [2]:
# Define the grid configuration and data center configuration for the simulation
ca_on_energy_profile = {
    GenerationType.SOLAR: 0.05,
    GenerationType.WIND: 0.08,
    GenerationType.HYDRO_RIVER: 0.24,
    GenerationType.NUCLEAR: 0.55,
    GenerationType.GAS: 0.8,
    GenerationType.BIOMASS: 0.008,
    GenerationType.OIL: 0.001,
}
energy_profile_share_descending = dict(sorted(ca_on_energy_profile.items(), key=lambda x: x[1], reverse=True))

grid_config = GridConfig(
        pp_grid = pn.case300(), # type: ignore
        region = GridRegion.CA_ON,
        energy_profile = energy_profile_share_descending
    )
data_center_config = DataCenterConfig(
    load_share = 0.5, 
    onsite_overprovision_factor = 1.0,
    onsite_generation_source = GenerationType.SOLAR,
)
load_id = grid_config.add_data_center(data_center_config)

# Initialize the grid simulation interface with the defined configurations
sim_interface = GridSimulation(
    grid_configs=[grid_config],
    shifting_threshold=10.0,
    enable_weather_variation=False
)

In [ ]:
# Verifies data center energy usage trend as load increases
start_date = pd.Timestamp("2024-01-01 00:00:00")
new_shares = [0.1 + 0.002 * i for i in range(400)]
SIMULATION_START_TIME: datetime = datetime.strptime('2020-08-01 00:00:00', '%Y-%m-%d %H:%M:%S')
current_time = SIMULATION_START_TIME

for share in new_shares:
    sim_interface.run_power_flow(date = current_time, observe = True)
    # sim_interface.update_dc_load_share("CA_ON", [load_id], [share])
    current_time += timedelta(hours = 1)

plot_data(
    x_data = new_shares,
    y_data = sim_interface.observer._ci_data,
    labels = sim_interface.observer._grid_names,
    x_label = "Data Center Load Share",
    y_label = "Grid Carbon Intensity (gCO2/kWh)",
    title = "Impact of Data Center Load Share on Grid Carbon Intensity"
)

SyntaxError: invalid syntax (4058136811.py, line 10)

In [ ]:
grid = sim_interface._grids[0]
energy_profile = grid.get_energy_profile()
for gen_type, share in energy_profile.items():
    print(f"{gen_type.value}: {share:.2%}")


solar: 0.00%
solar_non_varying: 0.00%
wind: 0.00%
wind_non_varying: 0.00%
coal: 0.00%
biomass: 0.00%
gas: 0.00%
oil: 0.00%
hydro_river: 0.00%
nuclear: 0.00%


In [ ]:
display(sim_interface._grids[0]._net.res_ext_grid)

,p_mw,q_mvar
0,-9.527033e-10,NaN


In [ ]:
display(sim_interface._grids[0]._net.res_gen)

,p_mw,q_mvar,va_degree,vm_pu
0,1.000000e+02,NaN,24.738501,1.0
1,1.167143e-09,NaN,25.320349,1.0
2,1.167143e-09,NaN,19.966375,1.0
3,1.167143e-09,NaN,-4.848584,1.0
4,1.167143e-09,NaN,-14.354720,1.0
5,3.925915e+02,NaN,-4.661780,1.0
6,1.136609e-09,NaN,-2.295790,1.0
7,3.079369e+02,NaN,6.362817,1.0
8,1.598755e-08,NaN,-9.185471,1.0
9,1.136609e-09,NaN,-12.950545,1.0


In [ ]:
display(sim_interface._grids[0]._net.ext_grid)

,name,bus,vm_pu,va_degree,slack_weight,in_service,max_p_mw,min_p_mw,max_q_mvar,min_q_mvar,controllable,p_mw,source_type
0,None,256,1.0507,0.0,1.0,True,0.0,-inf,10.0,0.0,True,0.0,coal


In [ ]:
display(sim_interface._grids[0]._net.gen)

,name,bus,p_mw,vm_pu,sn_mva,min_q_mvar,max_q_mvar,scaling,slack,in_service,slack_weight,type,controllable,max_p_mw,min_p_mw,id_q_capability_characteristic,reactive_capability_curve,curve_style,power_limit,source_type,is_onsite_gen,is_backup_gen
0,None,7,0.0,1.0153,NaN,-10.00,10.00,1.0,False,True,0.0,None,True,100.000,0.0,<NA>,False,None,100.000,oil,False,False
1,None,9,0.0,1.0205,NaN,-20.00,20.00,1.0,False,True,0.0,None,True,100.000,0.0,<NA>,False,None,100.000,coal,False,False
2,None,18,0.0,1.0010,NaN,-20.00,20.00,1.0,False,True,0.0,None,True,100.000,0.0,<NA>,False,None,100.000,coal,False,False
3,None,54,0.0,0.9583,NaN,-25.00,25.00,1.0,False,True,0.0,None,True,100.000,0.0,<NA>,False,None,100.000,coal,False,False
4,None,62,0.0,0.9632,NaN,12.00,35.00,1.0,False,True,0.0,None,True,100.000,0.0,<NA>,False,None,100.000,coal,False,False
5,None,68,0.0,1.0250,NaN,-240.00,240.00,1.0,False,True,0.0,None,True,475.000,0.0,<NA>,False,None,475.000,gas,False,False
6,None,75,0.0,1.0520,NaN,-11.00,96.00,1.0,False,True,0.0,None,True,255.000,0.0,<NA>,False,None,255.000,nuclear,False,False
7,None,76,0.0,1.0520,NaN,-153.00,153.00,1.0,False,True,0.0,None,True,390.000,0.0,<NA>,False,None,390.000,gas,False,False
8,None,79,0.0,1.0000,NaN,-30.00,56.00,1.0,False,True,0.0,None,True,168.000,0.0,<NA>,False,None,168.000,hydro_river,False,False
9,None,87,0.0,0.9900,NaN,-24.00,77.00,1.0,False,True,0.0,None,True,217.000,0.0,<NA>,False,None,217.000,nuclear,False,False


In [ ]:
display(sim_interface._grids[0]._net.sgen)

,name,bus,p_mw,q_mvar,sn_mva,scaling,in_service,type,current_source,controllable,id_q_capability_characteristic,reactive_capability_curve,curve_style,min_p_mw,max_p_mw,power_limit,source_type,is_onsite_gen,is_backup_gen
0,None,43,0.0,-5.00,NaN,1.0,True,,True,True,<NA>,False,None,0.0,5.0,5.0,coal,False,False
1,None,185,0.0,14.20,NaN,1.0,True,,True,True,<NA>,False,None,0.0,21.0,21.0,coal,False,False
2,None,228,0.0,17.00,NaN,1.0,True,,True,True,<NA>,False,None,0.0,23.0,23.0,coal,False,False
3,None,229,0.0,29.40,NaN,1.0,True,,True,True,<NA>,False,None,0.0,33.1,33.1,coal,False,False
4,None,233,0.0,-26.50,NaN,1.0,True,,True,True,<NA>,False,None,0.0,14.9,14.9,coal,False,False
5,None,238,0.0,1.40,NaN,1.0,True,,True,True,<NA>,False,None,0.0,11.1,11.1,coal,False,False
6,None,241,0.0,-76.70,NaN,1.0,True,,True,True,<NA>,False,None,0.0,113.7,113.7,solar,False,False
7,None,243,0.0,-34.17,NaN,1.0,True,,True,True,<NA>,False,None,0.0,100.0,100.0,coal,False,False


In [ ]:
display(sim_interface._grids[0]._net.load)

,name,bus,p_mw,q_mvar,const_z_p_percent,const_z_q_percent,const_i_p_percent,const_i_q_percent,sn_mva,scaling,in_service,type,controllable,is_data_center,power_limit
0,None,0,10707.59485,49.00,0.0,0.0,0.0,0.0,NaN,1.0,True,None,False,True,11923.825
1,None,1,56.00000,15.00,0.0,0.0,0.0,0.0,NaN,1.0,True,None,False,False,NaN
2,None,2,20.00000,0.00,0.0,0.0,0.0,0.0,NaN,1.0,True,None,False,False,NaN
3,None,4,353.00000,130.00,0.0,0.0,0.0,0.0,NaN,1.0,True,None,False,False,NaN
4,None,5,120.00000,41.00,0.0,0.0,0.0,0.0,NaN,1.0,True,None,False,False,NaN
5,None,7,63.00000,14.00,0.0,0.0,0.0,0.0,NaN,1.0,True,None,False,False,NaN
6,None,8,96.00000,43.00,0.0,0.0,0.0,0.0,NaN,1.0,True,None,False,False,NaN
7,None,9,153.00000,33.00,0.0,0.0,0.0,0.0,NaN,1.0,True,None,False,False,NaN
8,None,10,83.00000,21.00,0.0,0.0,0.0,0.0,NaN,1.0,True,None,False,False,NaN
9,None,12,58.00000,10.00,0.0,0.0,0.0,0.0,NaN,1.0,True,None,False,False,NaN


In [ ]:
display(sim_interface._grids[0]._net.poly_cost)

,element,et,cp0_eur,cp1_eur_per_mw,cp2_eur_per_mw2,cq0_eur,cq1_eur_per_mvar,cq2_eur_per_mvar2
0,0,gen,0.0,82.152381,0.0,0.0,0.0,0.0
1,1,gen,0.0,178.500000,0.0,0.0,0.0,0.0
2,10,gen,0.0,97.500000,0.0,0.0,0.0,0.0
3,11,gen,0.0,97.500000,0.0,0.0,0.0,0.0
4,12,gen,0.0,178.500000,0.0,0.0,0.0,0.0
5,13,gen,0.0,178.500000,0.0,0.0,0.0,0.0
6,14,gen,0.0,97.500000,0.0,0.0,0.0,0.0
7,15,gen,0.0,97.500000,0.0,0.0,0.0,0.0
8,16,gen,0.0,180.500000,0.0,0.0,0.0,0.0
9,17,gen,0.0,180.500000,0.0,0.0,0.0,0.0
